# Rule: **build_industrial_production_per_country**


**Description**

The industrial production is taken from the JRC-IDEES database. This dataset provides the detailed consumption of energy for various industrial processes. If the considered country is not part of the EU28, the energy consumption is taken from Eurostat. The industrial production is calculated for the reference year specified in the config file. The ammonia production is provided separately as a .csv file built by the rule `build_ammonia_production`.

After calculating industrial production, basic chemicals are disaggregated into ammonia, chlorine, methanol, and HVC. For chlorine, methanol, and HVC, national production is allocated by scaling their EU27 totals with a country-specific distribution key. This key represents each country’s share of the total EU27 chemical sector, and is intended to proxy the size of the country’s general chemical industry. The internal composition of the chemical sector is assumed to be the same across countries and to remain constant over time. This means chlorine, methanol, and HVC are assumed to follow the same relative pattern as the chemical sector as a whole (excluding ammonia), implying fixed implicit shares.

The following subcategories [kton/a] are considered:
- Electric arc
- Integrated steelworks
- Other chemicals
- Pharmaceutical products etc.
- Cement
- Ceramics & other NMM
- Glass production
- Pulp production
- Paper production
- Printing and media reproduction
- Food, beverages and tobacco
- Alumina production
- Aluminium - primary production
- Aluminium - secondary production
- Other non-ferrous metals
- Transport equipment
- Machinery equipment
- Textiles and leather
- Wood and wood products
- Other industrial sectors
- Ammonia
- HVC
- Chlorine
- Methanol

The configuration parameters associated with this rule are defined under the **industry** section of the config file:  
- industry.basic_chemicals_without_NH3_production_today
- industry.chlorine_production_today
- industry.methanol_production_today
- industry.HVC_production_today
- industry.reference_year

**Inputs**

- data/jrc_idees/archive/2024-05-20/EU27/`JRC-IDEES-2021_Industry_EU27.xlsx` 
- data/jrc_idees/archive/2024-05-20/{country}/`JRC-IDEES-2021_Industry_{country}.xlsx`
- resources/{prefix}/{name}/`ammonia_production.csv`
- data/data/`ch_industrial_production_per_subsector.csv`
- resources/{prefix}/{name}/`eurostat_energy_balances.csv`

Note: "archive" directories and database years may vary with different versions of PyPSA-EUR/PyPSA-Spain.

**Outputs**

- resources/{prefix}/{name}/`industrial_production_per_country.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

In [ ]:
##### Imports
import pandas as pd
import os 
import sys
import matplotlib.pyplot as plt

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Set options
pd.set_option("display.max_columns", None)

## `industrial_production_per_country.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_production_per_country.csv"

ind_prod_today = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

# Use the first column (country code) as DataFrame index
ind_prod_today = ind_prod_today.set_index(ind_prod_today.columns[0])

ind_prod_today.head()

See today's industrial production for a specific country

In [ ]:
# Select the country
country_code = "ES"

# Filter by country
ind_prod_today_country = ind_prod_today.loc[country_code]
ind_prod_today_country

In [ ]:
# Keep only numeric sector values for plotting
ind_prod_today_country_num = pd.to_numeric(ind_prod_today_country,
errors="coerce").dropna()

ax = ind_prod_today_country_num.plot(
     kind="bar",
     figsize=(12, 5),
     color="steelblue"
)
ax.set_title(f"Industrial production by sector ({country_code})")
ax.set_ylabel("Production (kton/a)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()